# An Assistant Answering from Company Documents (RAG)

An LLM doesn't know a company's internal documents: asked about them, it makes up a plausible answer. **Retrieval-augmented generation** (RAG) gives it, along with the question, the relevant excerpts of a document base.

Ingredients of this demonstration, all open and runnable on Colab:

- a small "assistant" LLM, [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B)
- a multilingual *embedding* model, which turns a text into a vector to search by meaning

A GPU is recommended (`Runtime > Change runtime type`). Without a GPU, the notebook loads the 0.6B version of the model instead: lighter, but its answers are much less reliable.

In [1]:
import warnings

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "1.7B" if device == "cuda" else "0.6B"  # 0.6B: light enough for a CPU

if device == "cpu":
    warnings.warn("No GPU found: the notebook loads the 0.6B model, whose answers are of lower quality. "
                  "For the 1.7B model, pick a GPU (Runtime > Change runtime type), then rerun the notebook.")

LLM = f"Qwen/Qwen3-{MODEL_SIZE}"
DTYPE = torch.float16 if device == "cuda" else torch.float32
EMBEDDER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map=device)
embedder = SentenceTransformer(EMBEDDER, device=device)
print(device, MODEL_SIZE)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

cuda 1.7B


## The Document Base

The (fictional) internal guide of the Acme company: each paragraph becomes an excerpt (*chunk*), prefixed by the title of its section.

In [2]:
DOCUMENT = """
Remote work
Permanent employees can work remotely up to 2 days per week, with their manager's approval.
Requests are made in the "Kiosk" HR tool, at least 15 days before the first remote day.
Acme provides a screen and an ergonomic chair to employees working remotely at least 1 day per week.

Leave
Each employee has 25 days of paid leave and 11 days off in lieu (RTT) per year.
Leave of more than 5 consecutive days must be booked in Kiosk at least 1 month in advance.

Expense reports
Meals during a business trip are reimbursed up to €23.50 per meal, with a receipt.
Expense reports must be filed in Kiosk within 45 days of the expense.
Reimbursement is paid with the salary of the month following approval.

Business travel
Train is mandatory for trips under 4 hours, in second class.
Beyond that, economy-class flights are allowed, with the department director's approval.
Hotel nights are reimbursed up to €140 in Paris and €95 elsewhere.

IT security
Two-factor authentication is mandatory on all company tools.
Copying customer data into a public AI tool is forbidden; use the internal assistant "AcmeGPT".
Any suspicious email must be forwarded to security@acme.example.

Training
Each employee is entitled to 3 days of training per year, on top of their personal training account.
Training requests are approved during the annual review.
"""

In [3]:
chunks = []
for section in DOCUMENT.strip().split("\n\n"):
    title, *paragraphs = section.strip().splitlines()
    chunks.extend(f"{title} — {paragraph}" for paragraph in paragraphs)
print(len(chunks))
chunks[:3]

16


["Remote work — Permanent employees can work remotely up to 2 days per week, with their manager's approval.",
 'Remote work — Requests are made in the "Kiosk" HR tool, at least 15 days before the first remote day.',
 'Remote work — Acme provides a screen and an ergonomic chair to employees working remotely at least 1 day per week.']

## Without RAG: the Model Makes Things Up

Let's plug the model in as an HR assistant, without giving it the guide, which it has never seen:

In [4]:
ROLE = "You are Acme's HR assistant. Answer employees' questions precisely and concisely."
QUESTION = "What is the reimbursement cap for a meal during a business trip?"

def ask(messages: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                           return_dict=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=400, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True)


print(ask([{"role": "system", "content": ROLE}, {"role": "user", "content": QUESTION}]))

The reimbursement cap for a meal during a business trip typically depends on the company's policies and local labor laws. Generally, the standard reimbursement for a meal is:

- **$15 per meal** (for a lunch or dinner)  
- **$3 per drink** (for a beverage)

These amounts are often subject to the following rules:

- **Lunch**: $15  
- **Dinner**: $15  
- **Drink**: $3  
- **Snacks**: Not typically reimbursed unless specified by the company

Please consult your company's HR policies or the employee handbook for the exact amount and conditions applicable to your situation.


The model cannot know the answer: if it gives an amount, it makes it up (the guide says €23.50). That is a **hallucination**. It sometimes admits it doesn't know, but you can't count on it.

## Retrieving the Relevant Excerpts

We compute the vector of every excerpt once and for all. For a question, we keep the excerpts whose vectors are closest to its own (similarity score between -1 and 1: the higher, the closer the meaning):

In [5]:
chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)


def retrieve(question: str, k: int = 3) -> list[tuple[float, str]]:
    question_embedding = embedder.encode(question, normalize_embeddings=True)
    scores = chunk_embeddings @ question_embedding
    best = scores.argsort()[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]


for score, chunk in retrieve(QUESTION):
    print(f"{score:.2f}  {chunk}")

0.78  Expense reports — Meals during a business trip are reimbursed up to €23.50 per meal, with a receipt.
0.55  Business travel — Hotel nights are reimbursed up to €140 in Paris and €95 elsewhere.
0.44  Expense reports — Reimbursement is paid with the salary of the month following approval.


## With RAG: Answering from the Excerpts

We give the model the retrieved excerpts, with the instruction to stick to them and cite its sources. The `rag` function also prints the retrieved excerpts and their score:

In [6]:
INSTRUCTIONS = ("Answer only from the provided excerpts, "
                "citing the number of the excerpt used. If the answer isn't there, "
                "say that you don't know.")
EXCERPTS_LABEL = "Excerpts:"
QUESTION_LABEL = "Question:"

def rag(question: str) -> str:
    results = retrieve(question)
    print("Retrieved excerpts (score, text):")
    for i, (score, chunk) in enumerate(results):
        print(f"[{i + 1}] {score:.2f}  {chunk}")
    print("\nAnswer:")
    excerpts = "\n".join(f"[{i + 1}] {chunk}" for i, (_, chunk) in enumerate(results))
    return ask([
        {"role": "system", "content": f"{ROLE} {INSTRUCTIONS}"},
        {"role": "user", "content": f"{EXCERPTS_LABEL}\n{excerpts}\n\n{QUESTION_LABEL} {question}"},
    ])


print(rag(QUESTION))

Retrieved excerpts (score, text):
[1] 0.78  Expense reports — Meals during a business trip are reimbursed up to €23.50 per meal, with a receipt.
[2] 0.55  Business travel — Hotel nights are reimbursed up to €140 in Paris and €95 elsewhere.
[3] 0.44  Expense reports — Reimbursement is paid with the salary of the month following approval.

Answer:
The reimbursement cap for a meal during a business trip is up to €23.50 per meal, with a receipt. This is stated in Excerpt [1].


## A Question Outside the Base

The guide says nothing about company cars. A good assistant must admit it rather than make something up:

In [7]:
OUTSIDE_QUESTION = "What is Acme's policy on company cars?"

print(rag(OUTSIDE_QUESTION))

Retrieved excerpts (score, text):
[1] 0.30  Business travel — Beyond that, economy-class flights are allowed, with the department director's approval.
[2] 0.30  IT security — Any suspicious email must be forwarded to security@acme.example.
[3] 0.27  Remote work — Acme provides a screen and an ergonomic chair to employees working remotely at least 1 day per week.

Answer:
I don't have information regarding Acme's policy on company cars.


Look at the displayed excerpts: the search always returns the `k` closest ones, even when they say nothing about the question. There is no score threshold. So it is the instruction "if the answer isn't there, say you don't know" that has to keep the model from making things up; discarding the excerpts below a minimum score would be an extra safeguard. A small model doesn't always follow the instruction: judge the answer displayed above.

## A Differently Worded Question

The search works by meaning, not by exact words: "lunch on a work trip" finds the excerpt about meals during business trips.

In [8]:
REPHRASED_QUESTION = "How much can I spend on lunch when I'm on a work trip?"

print(rag(REPHRASED_QUESTION))

Retrieved excerpts (score, text):
[1] 0.68  Expense reports — Meals during a business trip are reimbursed up to €23.50 per meal, with a receipt.
[2] 0.46  Business travel — Hotel nights are reimbursed up to €140 in Paris and €95 elsewhere.
[3] 0.34  Remote work — Permanent employees can work remotely up to 2 days per week, with their manager's approval.

Answer:
The reimbursement for meals during a business trip is up to €23.50 per meal, with a receipt. This is stated in Excerpt [1].


## Takeaways

- Without access to the documents, an LLM hallucinates confidently
- RAG combines a search by meaning with a generation guided by the retrieved excerpts
- Citing sources makes answers checkable
- Quality depends first on the document base (up to date, well split) and on the search: a bad search gives a bad answer
- Even with RAG, a small model can be wrong: evaluate on your own questions before deploying